# 銀行客戶流失預測與留存策略分析 (Bank Customer Churn & Retention Strategy)

## 1. 簡介

本報告旨在透過機器學習模型，預測銀行客戶流失（Churn）的可能性，並深入分析導致客戶流失的關鍵因素。透過這些洞察，銀行可以制定更具針對性的客戶留存策略，從而降低客戶流失率，提升客戶忠誠度與業務收益。

## 2. 數據加載與探索性數據分析 (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the dataset
df = pd.read_csv('bank_churn.csv')

# Basic Info
print("Dataset Info:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

# Data Cleaning
# Drop unnecessary columns
df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Summary Statistics
print("\nSummary Statistics:")
print(df_clean.describe())

# Check for missing values
print("\nMissing Values:")
print(df_clean.isnull().sum())

# Visualization Settings
plt.style.use('ggplot')
sns.set_palette("husl")

# 1. Target Variable Distribution (Churn)
plt.figure(figsize=(8, 6))
sns.countplot(x='Exited', data=df_clean)
plt.title('Distribution of Customer Churn (Exited)')
plt.savefig('churn_distribution.png')
plt.close()

# 2. Churn by Geography
plt.figure(figsize=(10, 6))
sns.countplot(x='Geography', hue='Exited', data=df_clean)
plt.title('Churn by Geography')
plt.savefig('churn_by_geography.png')
plt.close()

# 3. Churn by Gender
plt.figure(figsize=(10, 6))
sns.countplot(x='Gender', hue='Exited', data=df_clean)
plt.title('Churn by Gender')
plt.savefig('churn_by_gender.png')
plt.close()

# 4. Age Distribution by Churn
plt.figure(figsize=(10, 6))
sns.kdeplot(df_clean[df_clean['Exited'] == 0]['Age'], label='Stayed', fill=True)
sns.kdeplot(df_clean[df_clean['Exited'] == 1]['Age'], label='Exited', fill=True)
plt.title('Age Distribution by Churn')
plt.legend()
plt.savefig('age_distribution_by_churn.png')
plt.close()

# 5. Correlation Heatmap
plt.figure(figsize=(12, 10))
# Select only numeric columns for correlation
numeric_df = df_clean.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.savefig('correlation_heatmap.png')
plt.close()

print("\nEDA completed and visualizations saved.")


## 3. 機器學習模型與評估

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('bank_churn.csv')
df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encoding categorical variables
le = LabelEncoder()
df_clean['Gender'] = le.fit_transform(df_clean['Gender'])
df_clean = pd.get_dummies(df_clean, columns=['Geography'], drop_first=True)

# Feature and Target selection
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model Training - Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png')
plt.close()

# Feature Importance
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importance for Customer Churn')
plt.savefig('feature_importance.png')
plt.close()

# Save the feature importance for later use in reporting
feature_importance_df.to_csv('feature_importance.csv', index=False)

print("\nModel training and evaluation completed.")


## 4. 關鍵流失驅動因素與商業洞察

透過特徵重要性分析，我們識別出影響客戶流失的關鍵因素。這對於制定有效的留存策略至關重要。

![特徵重要性](feature_importance.png)

**主要流失驅動因素（依重要性排序）：**

| 特徵名稱          | 重要性 (Importance) |
| :---------------- | :------------------ |
| Age (年齡)        | 0.253               |
| Balance (帳戶餘額) | 0.143               |
| EstimatedSalary (預估薪資) | 0.139               |
| CreditScore (信用分數) | 0.135               |
| NumOfProducts (產品數量) | 0.129               |
| Tenure (客戶關係年限) | 0.080               |
| IsActiveMember (是否活躍會員) | 0.036               |
| Geography_Germany (德國客戶) | 0.033               |
| Gender (性別)     | 0.020               |
| HasCrCard (是否持有信用卡) | 0.018               |
| Geography_Spain (西班牙客戶) | 0.015               |

**商業洞察與建議：**

1.  **年齡是首要考量：** 年齡是影響客戶流失的最重要因素。銀行應特別關注 40-50 歲的客戶群體，這一年齡段的客戶流失風險較高。可以針對此群體推出專屬的理財產品、退休規劃服務或忠誠度計劃。
2.  **帳戶餘額與產品數量：** 帳戶餘額和產品數量也是重要的流失指標。對於餘額較低或僅擁有一兩種產品的客戶，銀行應主動提供個性化的產品推薦，鼓勵他們使用更多銀行服務，例如投資產品、貸款或保險，以增加客戶黏性。
3.  **信用分數與預估薪資：** 信用分數和預估薪資雖然重要性略低於年齡和餘額，但仍是客戶財務狀況的重要指標。銀行可以利用這些資訊，為高信用分數或高薪資潛力的客戶提供更優惠的服務或專屬權益，以提升其滿意度。
4.  **客戶關係年限：** 客戶關係年限較短的客戶可能更容易流失。銀行應加強對新客戶的關懷和引導，確保他們在初期能順利適應銀行服務，並建立良好的客戶關係。
5.  **活躍度與地理位置：** 不活躍的會員和德國地區的客戶流失風險較高。銀行可以針對不活躍會員推出重新激活活動，例如提供限時優惠或專屬服務。對於德國市場，銀行需要深入分析當地客戶的特殊需求和競爭環境，制定差異化的市場策略。

## 5. 總結

本報告透過數據分析和機器學習模型，成功識別了銀行客戶流失的關鍵驅動因素，並提供了具體的商業建議。透過實施這些策略，銀行有望有效降低客戶流失率，提升客戶滿意度和市場競爭力。未來的研究可以進一步探索更複雜的模型演算法，並結合更多外部數據（如宏觀經濟數據、社交媒體情緒）來提升預測的準確性。